# LaDIS example: generating synthetic Bot-IoT Keylogging traffic

This notebook shows how to use LaDIS with the Bot-IoT Keylogging scenario, starting from the original PCAP captures and the corresponding Bot-IoT ground-truth file.

The workflow is:

1. Extract features from the PCAP files and separate Keylogging and Benign traffic
2. Train LaDIS using only the Keylogging flows, and inspect the selected latent dimension and denoiser.
3. Generate new synthetic Keylogging observations.
4. Save and reload the fitted LaDIS model.

The repository does not redistribute Bot-IoT data. Download the original files from the dataset provider before running the PCAP extraction cells.

If a tabular feature file has already been extracted, the PCAP section can be skipped and LaDIS can be fitted directly from the CSV, Excel file, Parquet file, or pandas DataFrame.


## 1. Imports

PCAP extraction uses the optional LaDIS PCAP dependencies. From the repository root, install them with:

```bash
pip install -e ".[pcap]"
```


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from ladis.feature_extraction import PCAPFeatureExtractor, discover_pcap_files, extract_botiot_pcaps, split_botiot_traffic
from ladis.pipeline import LaDIS, build_ladis_smoke_test_config

## 2. Feature extraction from PCAP files

### 2.1. Project and data paths

Keep the downloaded dataset outside the Git repository. A convenient local structure is:

```text
Bot-IoT/
└── Keylogging/
    ├── pcaps/
    │   ├── <capture_1>.pcap
    │   ├── <capture_2>.pcap
    │   └── <capture_3>.pcap
    ├── ground_truth/
    │   └── Keylogging.csv
    └── outputs/
```

Change `DATA_ROOT` below to the local Keylogging directory.


In [ ]:
# Resolve the LaDIS repository root whether Jupyter was launched from the repository root or from the examples directory
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIG_PATH = PROJECT_ROOT / "configs" / "bot_iot_keylogging.yaml"
# CHANGE this path to the downloaded Bot-IoT Keylogging directory
DATA_ROOT = Path("/path/to/Bot-IoT/Keylogging")
PCAP_DIR = DATA_ROOT / "pcaps"
GROUND_TRUTH_PATH = DATA_ROOT / "ground_truth" / "Keylogging.csv"
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Preparation config:", CONFIG_PATH)
print("CUDA available:", torch.cuda.is_available())

### 2.2. Locate the Keylogging PCAP files

The original Keylogging scenario contains several PCAP captures. Every supported `.pcap`, `.pcapng`, or `.cap` file found in the selected directory can be processed.

In [ ]:
pcap_files = discover_pcap_files(PCAP_DIR)
print(f"PCAP files found: {len(pcap_files)}")
for pcap_path in pcap_files:
    print(" -", pcap_path.name)
if not pcap_files:
    raise FileNotFoundError(f"No supported PCAP files were found in {PCAP_DIR}")
if not GROUND_TRUTH_PATH.is_file():
    raise FileNotFoundError(f"Bot-IoT ground truth was not found: {GROUND_TRUTH_PATH}")

### 2.3. Extract flow-level features

Bot-IoT PCAPs mix benign and attack traffic. First, the common flow representation is extracted and, then, labels are assigned by matching each extracted bidirectional flow against the Bot-IoT ground truth.

The extraction settings used are:

- 30 s inactivity timeout.
- Maximum of 5,000 packets per active flow.
- Canonical bidirectional flow construction.
- 114 output columns including flow metadata, statistical descriptors, TCP flag counts, and the traffic label.


In [ ]:
extractor = PCAPFeatureExtractor(flow_timeout=30.0, max_packets_per_flow=5000, show_progress=True)
extraction_result = extract_botiot_pcaps(pcap_files, ground_truth_path=GROUND_TRUTH_PATH, scenario_label="Keylogging", output_path=None, extractor=extractor, verbose=True)
matched_data = extraction_result.data
matching_diagnostics = extraction_result.diagnostics
print("\nExtracted matched table:", matched_data.shape)
print("\nTraffic labels:")
print(matched_data["label"].value_counts(dropna=False))

## 2.4. Separate Keylogging and Benign traffic

Attack-specific LaDIS models are trained independently, so the matched flows are separated by their final label.

This produces one table containing only Keylogging traffic and another containing only Benign traffic.

In [ ]:
traffic_tables = split_botiot_traffic(matched_data, output_dir=OUTPUT_DIR)
for label, table in traffic_tables.items():
    print(f"{label}: {table.shape}")
keylogging_data = traffic_tables["Keylogging"].copy()
print("\nKeylogging table:")
print(keylogging_data.shape)
keylogging_data.head()

The extraction step writes the separated tables to `OUTPUT_DIR`, including:

```text
Keylogging_features.csv
Normal_features.csv
```

The remainder of this notebook uses only `Keylogging_features.csv`.


## 3. Training LaDIS

LaDIS provides the full model selection protocol used by the paper. That protocol is computationally expensive because it trains multiple autoencoder and latent-diffusion candidates.

For a quick demonstration, leave `RUN_FULL_PAPER_PROTOCOL = False`. This uses a deliberately reduced configuration that preserves the complete workflow but is intended only to demonstrate that the software works.

Set `RUN_FULL_PAPER_PROTOCOL = True` to use the full LaDIS configuration.


In [ ]:
RUN_FULL_PAPER_PROTOCOL = True
model_config = (None if RUN_FULL_PAPER_PROTOCOL else build_ladis_smoke_test_config())
print("Training mode:", "FULL LADIS PROTOCOL" if RUN_FULL_PAPER_PROTOCOL else "REDUCED DEMONSTRATION")

The preparation YAML contains the dataset-specific semantic roles and reconstruction rules for the 114-column Bot-IoT Keylogging representation.

The high-level `LaDIS.fit()` method performs the complete workflow, including data preparation, train/validation/test splitting, mixed-type preprocessing, latent-dimension selection, definitive autoencoder training, latent standardization, and denoiser selection.


In [ ]:
model = LaDIS(preparation_config_path=CONFIG_PATH, model_config=model_config, random_state=42, device=None, include_provenance=False, verbose=True)
model.fit(keylogging_data)

### 3.1. Inspect the selected LaDIS configuration


In [ ]:
print("Selected latent dimension:", model.selected_latent_dimension_)
print("Selected denoiser:", model.selected_diffusion_candidate_)
print("\nLatent-dimension ranking:")
display(model.latent_dimension_selection_["ranking"])
print("\nDenoiser comparison:")
display(model.denoiser_selection_["comparison"])

## 4. Generate new synthetic data

Choose the number of synthetic observations to generate. The output is reconstructed into the original tabular representation wherever reconstruction is well defined.

Observation-specific identifiers excluded during preparation are not fabricated. The original PCAP filename is not included in synthetic data because generated observations do not belong to any real capture file (`include_provenance=False`).


In [ ]:
NUM_SYNTHETIC_ROWS = 10_000
synthetic_keylogging = model.sample(NUM_SYNTHETIC_ROWS, random_state=42, include_provenance=False, validate=True)
print("Synthetic shape:", synthetic_keylogging.shape)
synthetic_keylogging.head()

### 4.1. Save the synthetic data


In [ ]:
SYNTHETIC_PATH = OUTPUT_DIR / "Keylogging_ladis.csv"
synthetic_keylogging.to_csv(SYNTHETIC_PATH, index=False)
print("Synthetic data saved to:")
print(SYNTHETIC_PATH)

### 4.2. Save the fitted LaDIS model

Saving the fitted model avoids retraining when additional synthetic samples are needed later.


In [ ]:
CHECKPOINT_PATH = OUTPUT_DIR / "ladis_keylogging.joblib"
model.save(CHECKPOINT_PATH)
print("LaDIS checkpoint saved to:")
print(CHECKPOINT_PATH)

## 5. Reload the model and generate more data


In [ ]:
loaded_model = LaDIS.load(CHECKPOINT_PATH, device=None, random_state=42, verbose=False)
additional_synthetic = loaded_model.sample(1_000, random_state=123, include_provenance=False, validate=True)
print("Additional synthetic shape:", additional_synthetic.shape)
additional_synthetic.head()

## 6. Starting directly from an existing feature table

PCAP extraction is optional. If the 114-column Keylogging feature table has already been extracted, the training workflow can start directly from that file:

```python
from ladis.pipeline import LaDIS

model = LaDIS(preparation_config_path="configs/bot_iot_keylogging.yaml")
model.fit("/path/to/Keylogging_features.csv")
synthetic = model.sample(10_000)
```

LaDIS can also start from other supported tabular files or from a pandas DataFrame. For a different feature schema, use a dataset-specific preparation configuration rather than the Keylogging configuration shown in this example.
